In [1]:
import re
import string
import warnings
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
      accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

from catboost import CatBoostClassifier
from catboost import CatBoostRegressor, Pool

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
from catboost import CatBoostRegressor, Pool
import holidays

In [2]:
PATH = "posts_vk.csv"

df = pd.read_csv(PATH, engine= "python")
df["text"] = df["text"].fillna("").astype(str)
df = df.query("text != ''")
print(df.shape)
display(df.head())
display(df.info())


(37425, 22)


,domain,owner_id,post_id,post_key,from_id,dt_msk,edited_dt_msk,text,views,likes,comments,reposts,reactions_total,is_pinned,marked_as_ads,is_donut,post_source_type,post_type,n_photos,cover_photo_url,all_photo_urls,post_url
0,literabook,-33874468,953135,-33874468_953135,-33874468,2026-04-02 11:33:58+03:00,NaN,"Интимнее, чем секс \nВсякий раз, когда мы гово...",31453,66,2,60,66.0,0,0,0,api,post,1,https://sun9-26.userapi.com/s/v1/ig2/N24sE2iTm...,"[""https://sun9-26.userapi.com/s/v1/ig2/N24sE2i...",https://vk.com/wall-33874468_953135
1,literabook,-33874468,953106,-33874468_953106,-33874468,2026-04-02 10:22:39+03:00,NaN,«В Япoнии тяжело быть жeной глaвы меcтной адми...,88292,612,9,301,612.0,0,0,0,api,post,1,https://sun9-65.userapi.com/s/v1/ig2/IWxPKVhWS...,"[""https://sun9-65.userapi.com/s/v1/ig2/IWxPKVh...",https://vk.com/wall-33874468_953106
2,literabook,-33874468,953054,-33874468_953054,-33874468,2026-04-02 08:46:20+03:00,NaN,Запомни три правила. ☝️☝️☝️\n \nПервое. Если т...,85588,323,4,151,323.0,0,0,0,api,post,2,https://sun1-94.userapi.com/s/v1/ig2/6tkrGWgDb...,"[""https://sun1-94.userapi.com/s/v1/ig2/6tkrGWg...",https://vk.com/wall-33874468_953054
3,literabook,-33874468,953015,-33874468_953015,-33874468,2026-04-02 07:16:36+03:00,NaN,"Итак, получается, у нас есть: 2 литра «Кока-Ко...",11774,217,2,132,217.0,0,0,0,api,post,2,https://sun9-24.userapi.com/s/v1/ig2/eWLc9yJPv...,"[""https://sun9-24.userapi.com/s/v1/ig2/eWLc9yJ...",https://vk.com/wall-33874468_953015
4,literabook,-33874468,952996,-33874468_952996,-33874468,2026-04-02 05:45:38+03:00,NaN,❗Стали известны заказчики беспрецендентного из...,269501,1839,291,1768,1839.0,0,0,0,api,post,5,https://sun1-84.userapi.com/s/v1/ig2/b0BmjLlT_...,"[""https://sun1-84.userapi.com/s/v1/ig2/b0BmjLl...",https://vk.com/wall-33874468_952996


<class 'pandas.core.frame.DataFrame'>
Index: 37425 entries, 0 to 39689
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   domain            37425 non-null  object 
 1   owner_id          37425 non-null  int64  
 2   post_id           37425 non-null  int64  
 3   post_key          37425 non-null  object 
 4   from_id           37425 non-null  int64  
 5   dt_msk            37425 non-null  object 
 6   edited_dt_msk     5043 non-null   object 
 7   text              37425 non-null  object 
 8   views             37425 non-null  int64  
 9   likes             37425 non-null  int64  
 10  comments          37425 non-null  int64  
 11  reposts           37425 non-null  int64  
 12  reactions_total   37388 non-null  float64
 13  is_pinned         37425 non-null  int64  
 14  marked_as_ads     37425 non-null  int64  
 15  is_donut          37425 non-null  int64  
 16  post_source_type  37425 non-null  object 
 17

None

In [4]:
def filter_out_iqr(group: pd.DataFrame) -> pd.DataFrame:
    q1 = group["views"].quantile(0.25)
    q3 = group["views"].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return group[(group["views"] >= lower) & (group["views"] <= upper)]


def clean_text_minimal(text: str) -> str:
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = text.strip()

    return text


def base_preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "Unnamed: 0" in df.columns:
        df = df.drop(columns="Unnamed: 0")

    df["text"] = df["text"].apply(clean_text_minimal)
    df["dt_msk"] = pd.to_datetime(df["dt_msk"], errors="coerce")

    years = df["dt_msk"].dt.year.dropna().astype(int).unique().tolist()
    if years:
        ru_holidays = holidays.Russia(years=years)
        df["is_holiday"] = df["dt_msk"].dt.date.apply(
            lambda x: int(x in ru_holidays) if pd.notna(x) else 0
        )
    else:
        df["is_holiday"] = 0

    if {"domain", "views"}.issubset(df.columns):
        df = (
            df.groupby("domain", group_keys=False)
            .apply(filter_out_iqr)
            .reset_index(drop=True)
        )

    df = df.query("text != ''").copy()

    df["target_views"] = np.log1p(df["views"])

    df["engagement_rate"] = (
        (df["likes"].fillna(0) + df["comments"].fillna(0) + df["reposts"].fillna(0))
        / df["views"].replace(0, np.nan)
        * 100
    )

    drop_cols = [
        "post_type",
        "is_donut",
        "post_url",
        "marked_as_ads",
        "post_source_type",
        "views",
        "likes",
        "comments",
        "reposts",
        "reactions_total",
        "owner_id",
        "post_key",
        "from_id",
        "date_unix",
        "edited_dt_msk",
        "age_days",
        "cover_photo_url",
        "all_photo_urls",
        "is_pinned"
    ]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df.reset_index(drop=True)
    return df

In [5]:
df = base_preprocessing(df)
df.head()

,domain,post_id,dt_msk,text,n_photos,is_holiday,target_views,engagement_rate
0,lentach,24937205,2026-04-02 11:19:31+03:00,Минцифры собирается сократить число мелких про...,1,0,12.101106,0.766924
1,lentach,24937094,2026-04-02 10:42:51+03:00,NASA впервые за 54 года отправила астронавтов ...,1,0,11.778991,0.831539
2,lentach,24936809,2026-04-01 21:48:33+03:00,Воздух и небо стали красными на острове Крит в...,4,0,11.508566,1.061629
3,lentach,24936731,2026-04-01 20:17:47+03:00,"В Италии женщина умерла после того, как её раз...",1,0,11.988855,0.933830
4,lentach,24936681,2026-04-01 18:44:36+03:00,Сегодня Apple исполнилось 50 лет.\nКомпания бы...,1,0,11.097592,0.701398


In [6]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    dt = pd.to_datetime(df["dt_msk"], errors="coerce")

    df["hour"] = dt.dt.hour.fillna(0).astype("int16")
    df["weekday"] = dt.dt.weekday.fillna(0).astype("int16")
    df["month"] = dt.dt.month.fillna(0).astype("int16")
    df["day"] = dt.dt.day.fillna(0).astype("int16")

    df["is_weekend"] = (df["weekday"] >= 5).astype("int8")
    df["is_workday"] = (df["weekday"] < 5).astype("int8")
    df["is_friday"] = (df["weekday"] == 4).astype("int8")

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
    df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

    df["n_photos"] = df["n_photos"].fillna(0).astype("int16")

    txt = df["text"].fillna("").astype(str)

    df["text_len_chars"] = txt.str.len().astype("int32")
    df["text_len_words"] = txt.str.split().str.len().fillna(0).astype("int32")
    df["num_lines"] = (txt.str.count("\n") + 1).astype("int16")

    df["num_exclam"] = txt.str.count("!").astype("int16")
    df["num_question"] = txt.str.count(r"\?").astype("int16")
    df["num_dots"] = txt.str.count(r"\.").astype("int16")
    df["num_commas"] = txt.str.count(",").astype("int16")
    df["num_colons"] = txt.str.count(":").astype("int16")

    df["num_hashtags"] = txt.str.count("#").astype("int16")
    df["num_links"] = (txt.str.count("http") + txt.str.count("vk.cc")).astype("int16")
    df["has_mention"] = txt.str.contains(r"\[club|\[id|@", regex=True, na=False).astype(
        "int8"
    )
    df["num_emojis"] = txt.str.count(r"[\U0001F300-\U0001FAFF]").astype("int16")

    letters = txt.str.findall(r"[A-Za-zА-Яа-яЁё]")
    caps = txt.str.findall(r"[A-ZА-ЯЁ]")
    df["caps_ratio"] = (
        (caps.str.len() / letters.str.len().replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )
    words = txt.str.findall(r"\w+", flags=re.UNICODE)
    digits = txt.str.findall(r"\d")
    caps_words = txt.str.findall(r"\b[A-ZА-ЯЁ]{2,}\b")

    df["unique_words_count"] = words.apply(
        lambda x: len(set(w.lower() for w in x))
    ).astype("int32")

    df["unique_words_ratio"] = (
        (df["unique_words_count"] / df["text_len_words"].replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )

    df["avg_word_len"] = words.apply(
        lambda x: np.mean([len(w) for w in x]) if len(x) else 0
    ).astype("float32")

    df["long_words_count"] = words.apply(lambda x: sum(len(w) >= 8 for w in x)).astype(
        "int16"
    )

    df["digit_count"] = digits.str.len().astype("int16")
    df["digit_ratio"] = (
        (df["digit_count"] / df["text_len_chars"].replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )

    df["uppercase_words_count"] = caps_words.str.len().astype("int16")

    sentence_count = txt.str.count(r"[.!?]+")
    df["sentence_count"] = np.where(
        df["text_len_chars"] > 0,
        sentence_count + 1,
        0,
    ).astype("int16")

    df["avg_sentence_len_words"] = (
        (df["text_len_words"] / df["sentence_count"].replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )

    df["ellipsis_count"] = txt.str.count(r"\.\.\.|…").astype("int16")
    df["repeat_punct_count"] = txt.str.count(r"[!?]{2,}").astype("int16")

    df["has_url"] = txt.str.contains(r"http|www|vk\.cc", regex=True, na=False).astype(
        "int8"
    )
    df["has_number"] = txt.str.contains(r"\d", regex=True, na=False).astype("int8")
    df["has_price"] = txt.str.contains(
        r"\d+\s?(₽|руб|р\b)|скидк|цена|бесплатно|акция",
        regex=True,
        case=False,
        na=False,
    ).astype("int8")

    df["starts_with_question"] = txt.str.match(
        r"^\s*[^a-zA-ZА-Яа-яЁё0-9]*.*\?", na=False
    ).astype("int8")
    df["starts_with_number"] = txt.str.match(r"^\s*\d", na=False).astype("int8")
    df["starts_with_emoji"] = txt.str.match(
        r"^\s*[\U0001F300-\U0001FAFF]", na=False
    ).astype("int8")

    cta_pattern = (
        r"пиши|смотри|читай|успей|переходи|жми|сохрани|подпишись|голосуй|оцени"
    )
    urgency_pattern = r"сегодня|сейчас|срочно|только сегодня|последний шанс|успей"

    df["has_call_to_action"] = txt.str.contains(
        cta_pattern,
        regex=True,
        case=False,
        na=False,
    ).astype("int8")

    df["has_urgency"] = txt.str.contains(
        urgency_pattern,
        regex=True,
        case=False,
        na=False,
    ).astype("int8")

    df["part_of_day"] = pd.cut(
        df["hour"],
        bins=[-1, 5, 11, 17, 23],
        labels=["night", "morning", "afternoon", "evening"],
    )

    df["text_len_bin"] = pd.cut(
        df["text_len_words"],
        bins=[-1, 5, 15, 30, 60, 10**9],
        labels=["very_short", "short", "medium", "long", "very_long"],
    )

    df["hour_weekday"] = df["hour"].astype(str) + "_" + df["weekday"].astype(str)
    df["domain_weekday"] = (
        df["domain"].fillna("unknown").astype(str) + "_" + df["weekday"].astype(str)
    )
    df["domain_part_of_day"] = (
        df["domain"].fillna("unknown").astype(str) + "_" + df["part_of_day"].astype(str)
    )
    df["weekend_hour"] = df["is_weekend"].astype(str) + "_" + df["hour"].astype(str)

    return df

In [7]:
data = make_features(df)
data.head()

,domain,post_id,dt_msk,text,n_photos,is_holiday,target_views,engagement_rate,hour,weekday,month,day,is_weekend,is_workday,is_friday,hour_sin,hour_cos,weekday_sin,weekday_cos,text_len_chars,text_len_words,num_lines,num_exclam,num_question,num_dots,num_commas,num_colons,num_hashtags,num_links,has_mention,num_emojis,caps_ratio,unique_words_count,unique_words_ratio,avg_word_len,long_words_count,digit_count,digit_ratio,uppercase_words_count,sentence_count,avg_sentence_len_words,ellipsis_count,repeat_punct_count,has_url,has_number,has_price,starts_with_question,starts_with_number,starts_with_emoji,has_call_to_action,has_urgency,part_of_day,text_len_bin,hour_weekday,domain_weekday,domain_part_of_day,weekend_hour
0,lentach,24937205,2026-04-02 11:19:31+03:00,Минцифры собирается сократить число мелких про...,1,0,12.101106,0.766924,11,3,4,2,0,1,0,0.258819,-9.659258e-01,0.433884,-0.900969,505,71,3,0,0,3,5,0,0,0,0,0,0.019139,65,0.915493,5.957747,26,5,0.009901,1,4,17.750000,0,0,0,1,0,0,0,0,0,0,morning,very_long,11_3,lentach_3,lentach_morning,0_11
1,lentach,24937094,2026-04-02 10:42:51+03:00,NASA впервые за 54 года отправила астронавтов ...,1,0,11.778991,0.831539,10,3,4,2,0,1,0,0.500000,-8.660254e-01,0.433884,-0.900969,519,80,3,0,0,5,2,0,0,0,0,0,0.060680,71,0.887500,5.108434,19,12,0.023121,3,6,13.333333,0,0,0,1,0,0,0,0,0,0,morning,very_long,10_3,lentach_3,lentach_morning,0_10
2,lentach,24936809,2026-04-01 21:48:33+03:00,Воздух и небо стали красными на острове Крит в...,4,0,11.508566,1.061629,21,2,4,1,0,1,0,-0.707107,7.071068e-01,0.974928,-0.222521,86,16,1,0,0,0,0,0,0,0,0,0,0.043478,16,1.000000,4.312500,2,0,0.000000,0,1,16.000000,0,0,0,0,0,0,0,0,0,0,evening,medium,21_2,lentach_2,lentach_evening,0_21
3,lentach,24936731,2026-04-01 20:17:47+03:00,"В Италии женщина умерла после того, как её раз...",1,0,11.988855,0.933830,20,2,4,1,0,1,0,-0.866025,5.000000e-01,0.974928,-0.222521,267,38,2,0,0,1,4,0,0,0,0,0,0.013514,37,0.973684,5.743590,12,2,0.007491,0,2,19.000000,0,0,0,1,0,0,0,0,0,0,evening,long,20_2,lentach_2,lentach_evening,0_20
4,lentach,24936681,2026-04-01 18:44:36+03:00,Сегодня Apple исполнилось 50 лет.\nКомпания бы...,1,0,11.097592,0.701398,18,2,4,1,0,1,0,-1.000000,-1.836970e-16,0.974928,-0.222521,126,19,2,0,0,1,1,0,0,0,0,0,0.090909,18,0.947368,5.578948,5,7,0.055556,0,2,9.500000,0,0,0,1,0,0,0,0,0,1,evening,medium,18_2,lentach_2,lentach_evening,0_18


In [8]:
def print_metrics_clf(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"{name}:")
    print(f"  Accuracy  = {acc:.4f}")
    print(f"  Precision = {prec:.4f}")
    print(f"  Recall    = {rec:.4f}")
    print(f"  F1        = {f1:.4f}")
    print()

In [9]:
num_cols = ['n_photos',
       'hour', 'weekday', 'month', 'day', 'is_weekend',
       'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'text_len_chars',
       'text_len_words', 'num_lines', 'num_exclam', 'num_question', 'num_dots',
       'num_commas', 'num_colons', 'num_hashtags', 'num_links', 'has_mention',
       'num_emojis', 'caps_ratio', 'unique_words_count',
       'unique_words_ratio', 'avg_word_len', 'long_words_count', 'digit_count',
       'digit_ratio', 'uppercase_words_count', 'sentence_count',
       'avg_sentence_len_words', 'ellipsis_count', 'repeat_punct_count',
       'has_url', 'has_number', 'has_price', 'starts_with_question',
       'starts_with_number', 'starts_with_emoji', 'has_call_to_action',
       'has_urgency', 'is_holiday','is_workday','is_friday']

text_col = "text"
target_col = "engagement_rate"
cat_cols = [
    "domain",
    "part_of_day",
    "text_len_bin",
    "hour_weekday",
    "domain_weekday",
    "domain_part_of_day",
    "weekend_hour"
]

In [10]:
from sklearn.model_selection import train_test_split
import pandas as pd

num_cols = ['n_photos',
       'hour', 'weekday', 'month', 'day', 'is_weekend',
       'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'text_len_chars',
       'text_len_words', 'num_lines', 'num_exclam', 'num_question', 'num_dots',
       'num_commas', 'num_colons', 'num_hashtags', 'num_links', 'has_mention',
       'num_emojis', 'caps_ratio', 'unique_words_count',
       'unique_words_ratio', 'avg_word_len', 'long_words_count', 'digit_count',
       'digit_ratio', 'uppercase_words_count', 'sentence_count',
       'avg_sentence_len_words', 'ellipsis_count', 'repeat_punct_count',
       'has_url', 'has_number', 'has_price', 'starts_with_question',
       'starts_with_number', 'starts_with_emoji', 'has_call_to_action',
       'has_urgency', 'is_holiday', 'is_workday', 'is_friday']

text_col = "text"
target_col = "engagement_rate"

cat_cols = [
    "domain",
    "part_of_day",
    "text_len_bin",
    "hour_weekday",
    "domain_weekday",
    "domain_part_of_day",
    "weekend_hour"
]

feature_cols = [text_col] + num_cols + cat_cols

X = data[feature_cols].copy()
y = data[target_col].copy()

X_train, X_temp, y_train_reg, y_temp_reg = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_valid, X_test, y_valid_reg, y_test_reg = train_test_split(
    X_temp, y_temp_reg, test_size=0.50, random_state=42
)
X = data[[text_col] + num_cols + cat_cols].copy()
y = data[target_col].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

domain_medians = y_train.groupby(X_train["domain"]).median()

y_train = (y_train.values > X_train["domain"].map(domain_medians).values).astype(int)
y_valid = (y_valid.values > X_valid["domain"].map(domain_medians).values).astype(int)
y_test = (y_test.values > X_test["domain"].map(domain_medians).values).astype(int)

In [14]:
print("Train:")
print(pd.Series(y_train).value_counts())
print(pd.Series(y_train).value_counts(normalize=True))
print()

print("Validation:")
print(pd.Series(y_valid).value_counts())
print(pd.Series(y_valid).value_counts(normalize=True))
print()

print("Test:")
print(pd.Series(y_test).value_counts())
print(pd.Series(y_test).value_counts(normalize=True))

Train:
0    12003
1    12002
Name: count, dtype: int64
0    0.500021
1    0.499979
Name: proportion, dtype: float64

Validation:
0    2600
1    2544
Name: count, dtype: int64
0    0.505443
1    0.494557
Name: proportion, dtype: float64

Test:
0    2679
1    2465
Name: count, dtype: int64
0    0.520801
1    0.479199
Name: proportion, dtype: float64


In [12]:
text_transformer = Pipeline([
    ("selector", FunctionTransformer(lambda x: x.squeeze(), validate=False)),
    ("tfidf", TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("tfidf", text_transformer, [text_col]),
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ],
    remainder="drop"
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("logreg", LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_valid = model.predict(X_valid)
y_pred_test = model.predict(X_test)

print_metrics_clf(y_train, y_pred_train, "Train")
print_metrics_clf(y_valid, y_pred_valid, "Validation")
print_metrics_clf(y_test, y_pred_test, "Test")

Train:
  Accuracy  = 0.8206
  Precision = 0.7995
  Recall    = 0.8558
  F1        = 0.8267

Validation:
  Accuracy  = 0.6851
  Precision = 0.6676
  Recall    = 0.7233
  F1        = 0.6943

Test:
  Accuracy  = 0.6905
  Precision = 0.6603
  Recall    = 0.7294
  F1        = 0.6931



In [13]:
from catboost import CatBoostClassifier, Pool

X = data[[text_col] + num_cols + cat_cols].copy()
y = data[target_col].copy()

# 70% train, 15% valid, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

domain_medians = y_train.groupby(X_train["domain"]).median()

y_train = (y_train > X_train["domain"].map(domain_medians)).astype(int)
y_valid = (y_valid > X_valid["domain"].map(domain_medians)).astype(int)
y_test = (y_test > X_test["domain"].map(domain_medians)).astype(int)

train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_cols,
    text_features=[text_col]
)

valid_pool = Pool(
    data=X_valid,
    label=y_valid,
    cat_features=cat_cols,
    text_features=[text_col]
)

test_pool = Pool(
    data=X_test,
    label=y_test,
    cat_features=cat_cols,
    text_features=[text_col]
)

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    loss_function="Logloss",
    eval_metric="F1",
    random_seed=42,
    verbose=100
)

model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True,
    early_stopping_rounds=100
)

y_pred_train = model.predict(train_pool).ravel()
y_pred_valid = model.predict(valid_pool).ravel()
y_pred_test = model.predict(test_pool).ravel()

print_metrics_clf(y_train, y_pred_train, "Train")
print_metrics_clf(y_valid, y_pred_valid, "Validation")
print_metrics_clf(y_test, y_pred_test, "Test")

0:	learn: 0.5736382	test: 0.5793291	best: 0.5793291 (0)	total: 231ms	remaining: 11m 33s
100:	learn: 0.6907779	test: 0.6764370	best: 0.6764370 (100)	total: 16.4s	remaining: 7m 50s
200:	learn: 0.7135074	test: 0.6925094	best: 0.6934334 (196)	total: 31.8s	remaining: 7m 23s
300:	learn: 0.7272581	test: 0.6960766	best: 0.6966967 (296)	total: 47.1s	remaining: 7m 2s
400:	learn: 0.7430331	test: 0.6990548	best: 0.7006393 (378)	total: 1m 2s	remaining: 6m 46s
500:	learn: 0.7641269	test: 0.6989186	best: 0.7012495 (464)	total: 1m 18s	remaining: 6m 31s
600:	learn: 0.7792723	test: 0.7022003	best: 0.7040758 (592)	total: 1m 34s	remaining: 6m 16s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7040758294
bestIteration = 592

Shrink model to first 593 iterations.
Train:
  Accuracy  = 0.7598
  Precision = 0.7437
  Recall    = 0.7930
  F1        = 0.7675

Validation:
  Accuracy  = 0.6965
  Precision = 0.6800
  Recall    = 0.7300
  F1        = 0.7041

Test:
  Accuracy  = 0.6788
  Precisio

In [15]:
from lightgbm import LGBMClassifier
preprocessor = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            Pipeline([
                ("selector", FunctionTransformer(lambda x: x.squeeze(), validate=False)),
                ("tfidf", TfidfVectorizer(
                    max_features=50000,
                    ngram_range=(1, 2),
                    min_df=3,
                    max_df=0.95,
                    sublinear_tf=True
                ))
            ]),
            [text_col]
        ),
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ],
    remainder="drop"
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("lgbm", LGBMClassifier(
        objective="binary",
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_valid = model.predict(X_valid)
y_pred_test = model.predict(X_test)

print_metrics_clf(y_train, y_pred_train, "Train")
print_metrics_clf(y_valid, y_pred_valid, "Validation")
print_metrics_clf(y_test, y_pred_test, "Test")

[LightGBM] [Info] Number of positive: 12002, number of negative: 12003
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.304184 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 238834
[LightGBM] [Info] Number of data points in the train set: 24005, number of used features: 10393
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499979 -> initscore=-0.000083
[LightGBM] [Info] Start training from score -0.000083
Train:
  Accuracy  = 0.9301
  Precision = 0.9096
  Recall    = 0.9551
  F1        = 0.9318

Validation:
  Accuracy  = 0.7039
  Precision = 0.6942
  Recall    = 0.7174
  F1        = 0.7056

Test:
  Accuracy  = 0.6903
  Precision = 0.6669
  Recall    = 0.7067
  F1        = 0.6862

